In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Upload cleaned data to Databricks

In [2]:
# Load Sales Data
sales_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("cleaned_sales.csv")

# Load Demand Data
demand_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("cleaned_demand.csv")

display(sales_df)
display(demand_df)

DataFrame[Region: string, Country: string, Item Type: string, Sales Channel: string, Order Priority: string, Order Date: date, Order ID: int, Ship Date: date, Units Sold: int, Unit Price: int, Unit Cost: int, Total Revenue: int, Total Cost: int, Total Profit: int, Profit Margin %: double]

DataFrame[ProductDemand-ID: string, Store ID: string, Total Price: int, Base Price: int, Units Sold: int, Discount %: double]

Add Profit Margin Column

In [3]:
from pyspark.sql.functions import col

sales_df = sales_df.withColumn(
    "Profit Margin %",
    (col("Total Profit") / col("Total Revenue")) * 100
)

Join Product + Sales Data

In [22]:
final_df = sales_df.join(
    demand_df,
    sales_df["Units Sold"] == demand_df["Units Sold"],
    "inner"
)

display(final_df)

DataFrame[Region: string, Country: string, Item Type: string, Sales Channel: string, Order Priority: string, Order Date: date, Order ID: int, Ship Date: date, Units Sold: int, Unit Price: int, Unit Cost: int, Total Revenue: int, Total Cost: int, Total Profit: int, Profit Margin %: double, ProductDemand-ID: string, Store ID: string, Total Price: int, Base Price: int, Units Sold: int, Discount %: double]

Calculate Final Metrics

In [5]:
from pyspark.sql.functions import sum

final_metrics = final_df.groupBy("Item Type") \
    .agg(
        sum("Total Revenue").alias("Total Revenue"),
        sum("Total Profit").alias("Total Profit")
    )

display(final_metrics)

DataFrame[Item Type: string, Total Revenue: bigint, Total Profit: bigint]

Save Output (Delta / CSV)

In [9]:
final_metrics.write.csv(
    "final_sales_metrics_csv",
    header=True,
    mode="overwrite"
)

Databricks SQL — Top 3 Best-Selling Products

In [23]:
final_df = final_df.select(
    col("Item Type"),
    col("Region"),
    sales_df["Units Sold"].alias("Units Sold"),
    col("Total Revenue"),
    col("Total Profit"),
    col("Store ID")
)
final_df.createOrReplaceTempView("sales_table")

In [24]:
result = spark.sql("""
SELECT
    s.`Item Type`,
    SUM(s.`Units Sold`) AS Total_Units_Sold
FROM sales_table s
GROUP BY s.`Item Type`
ORDER BY Total_Units_Sold DESC
LIMIT 3
""")

display(result)

DataFrame[Item Type: string, Total_Units_Sold: bigint]

In [30]:
result.show()


+-----------+----------------+
|  Item Type|Total_Units_Sold|
+-----------+----------------+
|       Food|             550|
|   Clothing|             520|
|Electronics|             360|
+-----------+----------------+

